In [10]:
import time
import requests
import pandas as pd

In [3]:
keey = '73bac766-deec-4df1-9f86-cfe53a5fce89'

In [4]:
headers = {'user-agent': 'ml-project'}

catgrs = {
    'Маркетплейсы, доставки, постаматы (100 м)': (100, 'постамат пункт выдачи'),
    'Медицинские уч. и аптеки (300 м)': (300, 'аптека поликлиника больница'),
    'Школы (300 м)': (300, 'школа'),
    'Остановки (300 м)': (300, 'остановки общественного транспорта'),
    'Продуктовые магазины (500 м)': (500, 'продуктовый магазин супермаркет'),
    'Пятерочки (500 м)': (500, 'Пятёрочка'),
}

In [5]:
def geocode(address):
  r = requests.get('https://catalog.api.2gis.com/3.0/items/geocode', params = {'q': address, 'fields': 'items.point', 'key': keey})
  point = r.json()['result']['items'][0]['point']
  return point['lat'], point['lon']

In [6]:
def count(lat, lon, radius, query):
  r = requests.get('https://catalog.api.2gis.com/3.0/items', params = {'q': query, 'point': f'{lon},{lat}', 'radius': radius, 'key': keey})
  return r.json().get('result', {}).get('total', 0)

In [7]:
def get_fs(address):
  lat, lon = geocode(address)
  result = {}
  for name, (radius, query) in catgrs.items():
    result[name] = count(lat, lon, radius, query)
    time.sleep(1)
  return result

In [9]:
fs = get_fs('Екатеринбург, улица Вайнера, 9')
for name, value in fs.items():
  print(name, value)

маркетплейсы, постаматы (100м) 2
медицина и аптеки (300м) 6
школы (300м) 1
остановки (300м) 2
продуктовые магазины (500м) 12
пятерочки (500м) 1


In [12]:
df = pd.read_csv('cian_commercial_150_700.csv')
df.columns.tolist()

['id', 'address', 'area_m2', 'url']

In [15]:
rows = []
for address in df['address']:
  try:
    fs = get_fs(address)
  except Exception:
    fs = {}
  fs['address'] = address
  rows.append(fs)
  print('на базе', address)

poi = pd.DataFrame(rows)

на базе Москва, ЦАО, р-н Пресненский, м. Краснопресненская, Рочдельская улица, 15С13
на базе Москва, ЦАО, р-н Пресненский, м. Краснопресненская, Рочдельская улица, 15С13
на базе Москва, ЦАО, р-н Пресненский, м. Краснопресненская, Рочдельская улица, 15С13
на базе Москва, ТАО (Троицкий), м. Апрелевка, д. Сенькино-Секерино, 82
на базе Москва, ВАО, р-н Гольяново, м. Черкизовская, Щелковское шоссе, 3с18
на базе Москва, НАО (Новомосковский), м. Прокшино, проспект Прокшинский, 5
на базе Москва, ВАО, р-н Соколиная гора, м. Шоссе Энтузиастов, шоссе Энтузиастов, 31С39
на базе Москва, ВАО, р-н Соколиная гора, м. Шоссе Энтузиастов, шоссе Энтузиастов, 31С39
на базе Москва, ВАО, р-н Соколиная гора, м. Шоссе Энтузиастов, шоссе Энтузиастов, 31С39
на базе Москва, ВАО, р-н Соколиная гора, м. Шоссе Энтузиастов, шоссе Энтузиастов, 31С39
на базе Москва, ВАО, р-н Соколиная гора, м. Шоссе Энтузиастов, шоссе Энтузиастов, 31С39
на базе Москва, ЮАО, р-н Даниловский, м. Павелецкая, 2-й Павелецкий проезд, 5С1
на 

In [17]:
poi.head()

,"маркетплейсы, постаматы (100м)",медицина и аптеки (300м),школы (300м),остановки (300м),продуктовые магазины (500м),пятерочки (500м),address
0,0.0,0.0,2.0,2.0,4.0,0.0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен..."
1,0.0,0.0,2.0,2.0,4.0,0.0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен..."
2,0.0,0.0,2.0,2.0,4.0,0.0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен..."
3,0.0,0.0,0.0,1.0,0.0,0.0,"Москва, ТАО (Троицкий), м. Апрелевка, д. Сеньк..."
4,0.0,5.0,0.0,2.0,8.0,1.0,"Москва, ВАО, р-н Гольяново, м. Черкизовская, Щ..."


In [21]:
poi = poi.dropna()

In [22]:
for f in poi.columns:
  if f != 'address':
    poi[f] = poi[f].astype(int)

/tmp/ipykernel_1219/3868368722.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  poi[f] = poi[f].astype(int)
/tmp/ipykernel_1219/3868368722.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  poi[f] = poi[f].astype(int)
/tmp/ipykernel_1219/3868368722.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

In [27]:
poi.to_csv('addresses_w_poi', index = False, encoding= 'utf-8-sig')